# Demand Planner: первые эксперименты

Прогноз продаж на семь дней и планирование закупок при ограниченном бюджете.

Это стартовый шаблон: данные, обучение и расчёты добавим по шагам. Выберите ядро **Python (demand-planner)**.

## 0. Окружение и пути

Запустите следующие две ячейки. Они проверяют окружение и определяют папки проекта.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from catboost import CatBoostRegressor


In [2]:
current_dir = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in [current_dir, *current_dir.parents]
     if (path / 'requirements.txt').is_file() and (path / 'src').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('Откройте ноутбук из папки demand-planner или notebooks.')
if Path(sys.prefix).resolve() != (PROJECT_ROOT / '.venv').resolve():
    raise RuntimeError('Выберите ядро Python (demand-planner) из окружения .venv.')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
RESULTS = PROJECT_ROOT / 'results'
RANDOM_STATE = 42
FORECAST_HORIZON = 7

pd.set_option('display.max_columns', 30)
print(f'Python: {sys.version.split()[0]}')
print(f'Проект: {PROJECT_ROOT}')
print(f'Данные: {DATA_RAW}')
print(f'Горизонт прогноза: {FORECAST_HORIZON} дней')


Python: 3.12.14
Проект: /Users/daniilsestov/Desktop/pet_projects/demand-planner
Данные: /Users/daniilsestov/Desktop/pet_projects/demand-planner/data/raw
Горизонт прогноза: 7 дней


## 1. Загрузка данных

Используем [M5 Forecasting — Accuracy](https://www.kaggle.com/competitions/m5-forecasting-accuracy/data), копия опубликована на [Zenodo](https://zenodo.org/records/12636070). Контрольная сумма архива сверяется с опубликованной.

Три исходные таблицы:

- `sales_train_validation.csv`: одна строка — товар в магазине; `d_1`, `d_2`, … содержат количество проданных единиц по дням.
- `calendar.csv`: связывает номера дней с датами, неделями и календарными событиями.
- `sell_prices.csv`: продажная цена товара в конкретном магазине за неделю. Это не закупочная стоимость.

Исходные CSV в `data/raw/` не меняем. Начнём с магазина **CA_1** и **100 товаров FOODS**. Выбор товаров случайный с фиксированным seed; будущие продажи не используются для ранжирования выборки.

In [ ]:
from src.data import download_m5, load_subset, build_panel

STORE_ID = 'CA_1'
CATEGORY = 'FOODS'
N_ITEMS = 100

# Если архив уже скачан, проверяем его и исходные файлы без повторной загрузки.
raw_files = download_m5(DATA_RAW)
display(raw_files)


In [ ]:
# Читаем большие CSV частями и оставляем выбранный магазин и товары.
calendar, sales, prices, source_shapes = load_subset(
    DATA_RAW, store_id=STORE_ID, category=CATEGORY,
    n_items=N_ITEMS, seed=RANDOM_STATE,
)
display(source_shapes)

subset_shapes = pd.DataFrame([
    {'table': 'calendar', 'rows': len(calendar), 'columns': calendar.shape[1]},
    {'table': 'sales: выбранные товары', 'rows': len(sales), 'columns': sales.shape[1]},
    {'table': 'prices: выбранные товары', 'rows': len(prices), 'columns': prices.shape[1]},
])
display(subset_shapes)


In [ ]:
# В sales дни пока лежат в отдельных столбцах: это широкий формат.
display(sales.iloc[:3, :12])
display(calendar.head(3))
display(prices.head(3))


## 2. Проверка и подготовка таблицы

Переводим продажи в длинный формат: одна строка — **один товар в одном магазине в один день**.

Сначала присоединяем календарь по `d`, затем цены по `store_id`, `item_id`, `wm_yr_wk`. Цена недельная, поэтому внутри недели она повторяется.

Нулевые продажи сохраняем: они не доказывают отсутствие спроса. Если товара не было в наличии, продажи тоже могли быть нулевыми; остатков в этих таблицах нет.

Отсутствие цены оставляем как пропуск. `price_available` означает наличие записи о цене, а не известный остаток товара. Пропуски в `event_name_*` ожидаемы для дней без отмеченного события. Цены из будущего не станут признаками автоматически: доступность информации на момент прогноза проверим на следующем этапе.

In [ ]:
panel = build_panel(calendar, sales, prices)

assert panel['item_id'].nunique() == N_ITEMS
assert panel['store_id'].nunique() == 1
assert panel['store_id'].eq(STORE_ID).all()
assert panel['cat_id'].eq(CATEGORY).all()

preview_columns = [
    'date', 'item_id', 'store_id', 'sales', 'sell_price',
    'price_available', 'weekday', 'event_name_1',
]
display(panel[preview_columns].head(10))


In [ ]:
data_summary = pd.Series({
    'rows': len(panel),
    'items': panel['item_id'].nunique(),
    'stores': panel['store_id'].nunique(),
    'days': panel['date'].nunique(),
    'date_from': panel['date'].min().date().isoformat(),
    'date_to': panel['date'].max().date().isoformat(),
    'duplicate_item_store_date': int(panel.duplicated(['item_id', 'store_id', 'date']).sum()),
    'missing_sales': int(panel['sales'].isna().sum()),
    'zero_sales_pct': round(100 * panel['sales'].eq(0).mean(), 2),
    'missing_price_rows': int(panel['sell_price'].isna().sum()),
    'positive_sales_without_price': int((panel['sales'].gt(0) & panel['sell_price'].isna()).sum()),
}, name='value')
display(data_summary.to_frame())

column_summary = pd.DataFrame({
    'dtype': panel.dtypes.astype(str),
    'missing': panel.isna().sum(),
})
display(column_summary)


In [ ]:
# Сохраняем локальную подготовленную таблицу; файлы данных исключены из Git.
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
processed_path = DATA_PROCESSED / 'm5_ca1_foods_100.parquet'
selected_items_path = DATA_PROCESSED / 'selected_items.csv'

panel.to_parquet(processed_path, index=False)
sales[['item_id', 'dept_id', 'cat_id', 'store_id']].sort_values('item_id').to_csv(
    selected_items_path, index=False,
)
print(f'Таблица сохранена: {processed_path.name}')
print(f'Выбранные товары: {selected_items_path.name}')


Теперь `panel` — наша основная таблица. Пока здесь нет признаков для обучения и целевой переменной следующей недели. В разделе 3 исследуем ряды, затем определим моменты прогнозирования и построим первый простой прогноз.

## 3. Исследование продаж

Изучим общий уровень продаж, различия между товарами и календарные закономерности. Здесь будут первые графики.

## 4. Простой недельный прогноз

Первый ориентир: продажи следующей недели равны продажам предыдущей. Сохраним прогноз и измерим ошибку.

## 5. Проверка по времени

Определим моменты прогнозирования и последовательные проверочные периоды. Все признаки и обучающие ответы должны быть известны на момент каждого прогноза.

## 6. CatBoost и анализ ошибок

Построим признаки из истории и сравним модель с простым прогнозом на одинаковых периодах. Сначала реализуем предыдущие разделы.

## 7. Результаты и следующие шаги

Запишем проверенные выводы, ограничения и идеи следующих экспериментов. Планирование закупок и AI-аналитика добавим после рабочего прогноза.